In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

### Schema Evolution
1. Adding New Columns (Manual / Automatic)

2. Widening Data Types (Supported Delta >= 3.2): Sometimes we need to expand a column's data type to accommodate larger values. Delta Lake allows "widening" type conversions that won't lose data, such as:
- `INT` to `BIGINT`
- `FLOAT` to `DOUBLE`
- `VARCHAR(10)` to `VARCHAR(20)`

3. Nested Structure Evolution (Manual / Automatic): Delta Lake supports evolution of complex data types like structs and arrays. We can:
- Add new fields to structs
- Modify nested field types
- Add new elements to arrays

4. Column Position Changes (Manual / Automatic): we can reorganize our columns

### Quick Note To Remember:
- `INSERT` works by matching columns by position
- `MERGE` works by matching columns by name

### Detail Summary On Schema Evolution in Delta Lake

#### 📌 Core Architectural Meaning

While **Schema Validation** is the aggressive gatekeeper that blocks unexpected changes, **Schema Evolution** is the feature that allows a table to safely grow, change, and adapt over time. It allows you to update a table's schema dynamically to accommodate changing business requirements *without* having to delete the table, rewrite historical data, or recreate your pipelines.

---

#### 📋 The 4 Drivers of Schema Evolution

##### 1. Adding New Columns (Expanding the Blueprint)

* **The Meaning:** The most common form of evolution. It happens when an upstream application introduces a brand-new field that your table needs to store moving forward.
* **The Manual Way:** You explicitly execute a DDL command (`ALTER TABLE ... ADD COLUMN`) to update the table’s metadata blueprint manually before running your data stream.
* **The Automatic Way:** You pass the explicit operational parameter **`.option("mergeSchema", "true")`** inside a PySpark write stream, or set the SQL environment configuration property `spark.databricks.delta.schema.autoMerge.enabled = true`. Delta will automatically detect the new column, append it to the log metadata, fill historical rows with `NULL`, and write the data cleanly.

##### 2. Type Widening (Scaling Data Capacity)

* **The Meaning:** Introduced in newer Delta Lake versions, this allows you to safely upgrade a column's data type to a larger, compatible container if your numbers grow too big.
* **The Rule:** You cannot randomly change a data type from an `INT` to a `STRING` because that would corrupt the underlying Parquet files. However, you can widen a column from **`INT` to `BIGINT**` (or `FLOAT` to `DOUBLE`) when your transactions span beyond ordinary integer limits.
* **The Mechanics:** Once enabled via table properties (`delta.enableTypeWidening = true`), you alter the column type. Delta gracefully maps the wider data type in the log metadata layer, allowing old files to coexist safely with newly appended wide files.

##### 3. Nested Structure Evolution (Managing Complex Schemas)

* **The Meaning:** Handles changes occurring inside complex, multi-layered data columns like `STRUCT` types (e.g., a `purchase_details` column that groups nested fields like `mall_pin_code` and `store_code`).
* **The Rule:** Just like flat schemas, nested structs can evolve. You can manually alter or use automatic schema merging to inject a brand-new sub-attribute (like `staff_id`) right into the middle of an existing struct container. Delta maintains absolute consistency by automatically assigning `NULL` to the new inner sub-attribute across all older historical records.

##### 4. Column Position Changes (Reordering Layouts)

* **The Meaning:** Dictates where a newly introduced column physically or logically falls inside your table sequence (e.g., forcing a new `age` column to sit explicitly *after* `price` rather than automatically getting dumped at the very end).
* **The Rule:** Standard positional `INSERT` statements will crash or mismatch if you change the expected order. However, if you utilize a robust **`MERGE INTO`** statement or write using data frames with schema merging active, Delta completely bypasses the positioning trap. It resolves the layout **by name**, appends the new column exactly where specified in the metadata index, and properly shifts the logical visibility schema layout for end-users.

---

#### 📊 Interview-Ready Summary

* **Schema Evolution** is Delta Lake's mechanism for gracefully adapting table frameworks to changes in upstream data streams without requiring resource-heavy table rewrites. By explicitly enabling properties like **`mergeSchema = true`** or utilizing native DDL alterations, developers can dynamically append new columns, evolve complex nested `STRUCT` elements, change logical column positioning, and execute **Type Widening** (such as `INT` to `BIGINT`) while completely preserving operational uptime and data lineage stability.

### Creating New Delta Table For This Lab

In [0]:
%sql
SELECT
  customer_id,
  invoice_no,
  price,
  invoice_date
FROM
  PARQUET.`abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/invoices/invoices_1_100.parquet`
WHERE
  customer_id BETWEEN 1 AND 5;

In [0]:
%sql
-- In prevous notebooks(i.e NB1, NB2), we created delta table using parquet files directly with CTAS.
-- Here we are first creating a schema for the delta table and then inserting data into it.

CREATE OR REPLACE TABLE delta_catalog.delta_db.invoices_se (
  customer_id INT NOT NULL,
  invoice_no STRING,
  price FLOAT,
  invoice_date DATE
);

-- Used parquet files has many columns but we are using the below ones only.
INSERT INTO delta_catalog.delta_db.invoices_se
  SELECT
    customer_id,
    invoice_no,
    price,
    invoice_date
  FROM
    PARQUET.`abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/invoices/invoices_1_100.parquet`
  WHERE
    customer_id BETWEEN 1 AND 5;

In [0]:
%sql
-- DROP TABLE delta_catalog.delta_db.invoices_se;
-- UNDROP TABLE delta_catalog.delta_db.invoices_se;

In [0]:
%sql
SELECT
  *
FROM
  delta_catalog.delta_db.invoices_se;

In [0]:
%sql
DESCRIBE HISTORY delta_catalog.delta_db.invoices_se;

In [0]:
%sql
SELECT
  *
FROM
  PARQUET.`abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/invoices/invoices_1_100.parquet`
ORDER BY price DESC;

### Scenario 1: Adding New Columns (Manual/Automatic)

#### Manual Way Of Adding New Columns

In [0]:
%sql
DESCRIBE TABLE delta_catalog.delta_db.invoices_se;

In [0]:
%sql
ALTER TABLE delta_catalog.delta_db.invoices_se
ADD COLUMN quantity INT;

In [0]:
%sql
DESCRIBE TABLE delta_catalog.delta_db.invoices_se;

In [0]:
%sql
INSERT INTO delta_catalog.delta_db.invoices_se
  SELECT
    customer_id,
    invoice_no,
    price,
    invoice_date,
    quantity
  FROM
    PARQUET.`abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/invoices/invoices_1_100.parquet`
  WHERE
    customer_id BETWEEN 6 AND 10;

In [0]:
%sql
-- Query returns the appropriate results with new records for customer_id 6 to 10, containing data for all old columns and new column quantity and nulls for historical quantity records, which means manual schema evolution worked for delta table.

SELECT
  *
FROM
  delta_catalog.delta_db.invoices_se
ORDER BY customer_id;

In [0]:
%sql
DESCRIBE HISTORY delta_catalog.delta_db.invoices_se;

#### Automatic Way Of Adding New Columns Using INSERT INTO

---

### Core Concept: Automatic Schema Evolution

**Definition:** The ability of a Delta Lake table to dynamically update its metadata structural blueprint to accommodate newly added or changed columns from incoming data streams, without requiring manual `ALTER TABLE` DDL commands or heavy table rewrites.

---

#### 1) SQL Implementation (Requires User-Managed Cluster)

##### 📝 The Technical "Why" (How it works under the hood)

This approach relies on a **Global Spark Session Configuration** (`SET spark.databricks...`). When you run this command, you are telling the entire underlying Apache Spark execution engine to globally modify its validation rules for the remainder of your notebook session.

Because traditional SQL `INSERT INTO` statements match data fields strictly by **ordinal position (left-to-right alignment)** rather than column names, turning on this session flag tells the compiler: *"If you see an extra column at the end of a positional insert, don't throw a structural mismatch error; instead, dynamically expand the target schema metadata."*

##### 🛑 Why it FAILS on Serverless Compute

* **Environment Sandboxing (Shared Access Mode):** Databricks Serverless clusters are multi-tenant and highly secure. To ensure cluster stability and isolate users, Serverless operates in **Shared Access Mode**, which explicitly blocks users from altering global Spark session environments via `SET` commands.
* **The Column Mapping Guardrail:** By default, Serverless turns on advanced **Column Mapping** (mapping logical column names to internal immutable physical IDs). Because Column Mapping demands strict, name-resolved mapping to assign these internal IDs, the engine completely bars blind positional SQL `INSERT` statements from altering the schema layout to prevent accidental structure corruption.

---

#### 2) Python / PySpark Implementation (Universal: Serverless & User-Managed)

##### 📝 The Technical "Why" (How it works under the hood)

This approach shifts the configuration from the global cluster scope down to the **DataFrame Writer scope** using `.option("mergeSchema", "true")`. Instead of altering the entire Spark platform environment, you are passing an isolation property tied strictly to that single write transaction.

The PySpark DataFrame API natively processes records using **Name-Based Evaluation**. When saving a table, the engine does not care about the left-to-right order of the columns in your file. It explicitly matches the text labels of the DataFrame columns against the target table's metadata attributes. If it detects a new column name like `payment_method`, it seamlessly allocates a new entry in the Delta transaction log (`_delta_log/`) and writes the data safely.

##### 🟢 Why it WORKS Natively on Serverless

* **Zero Shared Impact:** Because `.option()` changes the property of the specific *write stream* and not the *global cluster*, it complies perfectly with Serverless security boundaries.
* **Explicit ID Assignment:** Because the DataFrame contains explicit string column names, the modern Serverless Column Mapping engine can easily read the name, instantly generate a fresh hidden physical ID for the new column, and safely update the metadata wrapper without a single collision.

---

#### 📊 Summary Cheat Sheet for Your Notebook Review

| Metric / Feature | 1) SQL Positional Ingestion | 2) PySpark DataFrame API |
| --- | --- | --- |
| **Configuration Scope** | **Global / Session Level** (Alters entire cluster rule). | **Local / Operational Level** (Isolated to specific table write). |
| **Attribute Evaluation** | **Ordinal Position** (Left-to-right matrix matching). | **Name-Based Matching** (Resolves mapping explicitly by column label). |
| **Serverless Compatibility** | ❌ **Blocked** (Throws environment configuration exceptions). | 🟢 **Native Support** (The enterprise standard for portable pipelines). |

---

- To do **automatic Schema Evolution using pure SQL in Databricks**, you don't use an `.option()` configuration. Instead, you change a **Session Configuration Variable** right before you run your ingestion queries (`INSERT` or `MERGE`).
- **`.option("mergeSchema", "true")`** in PySpark is **scoped to a single target table write**. It is very safe because it only affects that specific dataframe pipeline.
- **`SET spark.databricks.delta.schema.autoMerge.enabled = true;`** in SQL is **global for your entire session**. If you have 5 different `INSERT` queries running in the same notebook after this command, *all* of them will dynamically change their target tables if a column structure shifts.
- **Best Practice Advice:** Always remember to turn it back off (`SET spark.databricks.delta.schema.autoMerge.enabled = false;`) at the end of your specific data loading block so you don't accidentally evolve other tables by mistake!

##### Python Code (Works On Serverless and User-Managed Cluster Both)

In [0]:
# 1. Define your cloud storage path variables
source_parquet_path = "abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/invoices/invoices_1_100.parquet"
target_table_name = "delta_catalog.delta_db.invoices_se"

# 2. Read the raw parquet file into a Spark DataFrame
df_source = (spark.read
             .format("parquet")
             .load(source_parquet_path)
             .select("customer_id",
                    "invoice_no",
                     col("price").cast("float"),
                    "invoice_date",
                    "quantity",
                    "payment_method") # New column for automatic schema evolution.
             .filter("customer_id BETWEEN 16 AND 20"))
# display(df_source)

# 3. Write data with explicit Schema Evolution option.
(df_source.write
 .format("delta")
 .mode("append")
 .option("mergeSchema", "true") # 👈 The golden alternative to the SQL SET command
 .saveAsTable(target_table_name))

In [0]:
%sql
-- Query returns the appropriate results with new record from 16-20 customer_id, containing data for all old columns and new column payment_method and nulls for historical payment_method records, which means automatic schema evolution worked.

SELECT
  *
FROM
  delta_catalog.delta_db.invoices_se
ORDER BY customer_id;

##### SQL Equivalent Code (Not For Serverless / Need User-Managed Cluster To Work)

In [0]:
%sql
-- option 1
-- Enables auto-merge at global level
-- Session scoped
-- The SQL alternative to python code .option("mergeSchema", "true").
SET spark.databricks.delta.schema.autoMerge.enabled = true; --This setting is not supported in serverless environments, For safety reasons as it does it at global level.

-- option 2
-- 1. Enable auto-merge strictly at the table metadata level (For Serverless if above is not working)
-- Table scoped
ALTER TABLE delta_catalog.delta_db.invoices_se
SET TBLPROPERTIES ('spark.databricks.delta.schema.autoMerge.enabled' = 'true');

In [0]:
%sql
DESCRIBE EXTENDED delta_catalog.delta_db.invoices_se;

-- Check the 'Detailed Table Information' section to verify the setting is set to true.

In [0]:
%sql
-- Note :- This cell needs user-managed cluster and setting SET spark.databricks.delta.schema.autoMerge.enabled = true;

INSERT INTO delta_catalog.delta_db.invoices_se 
  SELECT
    customer_id,
    invoice_no,
    price,
    invoice_date,
    quantity,
    payment_method,
    age -- New column for automatic schema evolution.
  FROM
    PARQUET.`abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/invoices/invoices_1_100.parquet`
  WHERE
    customer_id BETWEEN 11 AND 15;

In [0]:
%sql
-- Query returns the appropriate results with new record from 11-15 customer_id, containing data for all old columns and new column age and nulls for historical age records, which means automatic schema evolution worked.

SELECT
  *
FROM
  delta_catalog.delta_db.invoices_se
ORDER BY customer_id;

In [0]:
%sql
DESCRIBE HISTORY delta_catalog.delta_db.invoices_se;

#### Automatic Way Of Adding New Columns Using MERGE INTO

In [0]:
%sql
DESCRIBE TABLE delta_catalog.delta_db.invoices_se;

In [0]:
%sql
-- 'WITH SCHEMA EVOLUTION' Enables upsert and automatically add new columns on the fly.
MERGE WITH SCHEMA EVOLUTION INTO delta_catalog.delta_db.invoices_se AS target
USING (
  SELECT
    customer_id,
    invoice_no,
    price,
    current_date() AS invoice_date, 
    quantity,
    payment_method,
    age,                -- if above sql code for automatic schema evolution using 'Insert Into' is not executed, then this will also be a new column.
    shopping_mall       -- New column for automatic schema evolution.
  FROM
    PARQUET.`abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/invoices/invoices_1_100.parquet`
  WHERE
    customer_id BETWEEN 19 AND 25
) AS source
ON target.customer_id = source.customer_id
WHEN MATCHED THEN       -- If customer exists, overwrite all fields with fresh data & adds new column.
  UPDATE SET * 
WHEN NOT MATCHED THEN   -- If customer is brand new, insert the entire row.
  INSERT *; 

In [0]:
%sql
-- Query returns the appropriate results with updated and new record from 19-25 customer_id, containing data for all old columns and new column shopping_mall(And Age Maybe) and nulls for historical shopping_mall(And Age Maybe) records, which means automatic schema evolution worked.

SELECT
  *
FROM
  delta_catalog.delta_db.invoices_se
ORDER BY customer_id;

In [0]:
%sql
DESCRIBE HISTORY delta_catalog.delta_db.invoices_se;

### Scenario 2: Data Types Widening

#### Manual Way Of Data Types Widening

In [0]:
%sql
-- Enabling Type Widening strictly at the table level.
-- Table scoped
ALTER TABLE delta_catalog.delta_db.invoices_se
SET TBLPROPERTIES ('delta.enableTypeWidening' = 'true');

In [0]:
%sql
DESCRIBE EXTENDED delta_catalog.delta_db.invoices_se;

-- Check the 'Detailed Table Information' section to verify the setting is set to true.

In [0]:
%sql
ALTER TABLE delta_catalog.delta_db.invoices_se
ALTER COLUMN customer_id TYPE LONG;

In [0]:
%sql
DESCRIBE Table delta_catalog.delta_db.invoices_se;
-- manual type widening worked customer_id column changed from int to long.

In [0]:
%sql
INSERT INTO delta_catalog.delta_db.invoices_se
VALUES(987654321012345, 'I8888', 100, '2022-01-01', 33, 'Credit Card', "22", "Mall of Istanbul");

In [0]:
%sql
-- Query returns the appropriate results with customer_id = 987654321012345.

SELECT
  *
FROM
  delta_catalog.delta_db.invoices_se
WHERE
  customer_id = 987654321012345;    

#### Automatic Way Of Data Types Widening

In [0]:
%sql
-- Enable type widening feature on the table ('delta.enableTypeWidening' = 'true').
-- Execute Merge with Schema Evolution
MERGE WITH SCHEMA EVOLUTION INTO
  delta_catalog.delta_db.invoices_se AS target
USING (
  SELECT
    customer_id,
    invoice_no,
    price, -- automatic type widening price from float to double.
    date_sub(current_date(), 10) AS invoice_date,
    quantity,
    payment_method,
    age,
    shopping_mall
  FROM
    PARQUET.`abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/invoices/invoices_1_100.parquet`
  WHERE
    customer_id BETWEEN 30 AND 35
) AS source
ON
  target.customer_id = source.customer_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
# Enable type widening feature on the table ('delta.enableTypeWidening' = 'true').
# Load data, fix customer_id to match target, and let quantity widen automatically.
df_incoming = (
    spark.read.format("parquet")
    .load(
        "abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/invoices/invoices_1_100.parquet"
    )
    .filter(col("customer_id").between(40, 43))
    .selectExpr(
        "CAST(customer_id AS LONG) AS customer_id",  #added as we manually type widening customer_id from int to bigint in target table
        "invoice_no",
        "price",
        "date_sub(current_date(), 15) AS invoice_date",
        "CAST(quantity AS LONG) AS quantity",  # automatic type widening quantity from int to bigint
        "payment_method",
        "age",
        "shopping_mall",
    )
)
# display(df_incoming)

# 3. Save using mergeSchema
(
    df_incoming.write.format("delta")
    .option("mergeSchema", "true")
    .mode("append")
    .saveAsTable("delta_catalog.delta_db.invoices_se")
)

In [0]:
%sql
SELECT
  *
FROM
  delta_catalog.delta_db.invoices_se; 

-- check the price column datatype to verify as automatic type widening changed the type from float to double 

In [0]:
%sql
DESCRIBE Table delta_catalog.delta_db.invoices_se;

-- Automatic type widening worked with sql and pyspark using merge into.
-- price column changed from float to double.
-- And quantity column changed from int to long.

In [0]:
%sql
DESCRIBE HISTORY delta_catalog.delta_db.invoices_se;

### Scenario 3: Nested Structure Evolution (Manual / Automatic)

#### Manual Nested Structure Evolution

In [0]:
%sql
ALTER TABLE
  delta_catalog.delta_db.invoices_se
ADD COLUMN
  purchase_details
  STRUCT<mall_pin_code INT, store_code INT>;

In [0]:
%sql
-- target table columns

-- customer_id
-- invoice_no
-- price
-- invoice_date
-- quantity
-- payment_method
-- age
-- shopping_mall
-- purchase_details

INSERT INTO delta_catalog.delta_db.invoices_se
VALUES(50, 'I0000', 100, '2022-01-01', 33, 'Credit Card', "22", "Mall of Istanbul", struct(12345, 1000));  

In [0]:
%sql
-- Query returns the appropriate results with data for customer_id = 50 and newly created struct column purchase_details.

SELECT
  *
FROM
  delta_catalog.delta_db.invoices_se
ORDER BY
  customer_id;

In [0]:
%sql
DESCRIBE TABLE delta_catalog.delta_db.invoices_se;

In [0]:
%sql
-- Manual Type widening For Struct Type
ALTER TABLE delta_catalog.delta_db.invoices_se
ALTER COLUMN purchase_details.mall_pin_code TYPE LONG;

In [0]:
%sql
INSERT INTO delta_catalog.delta_db.invoices_se
VALUES(51, 'I0001', 100, '2026-06-19', 33, 'Credit Card', "23", "Mall of Istanbul", struct(987654321012345, 12345));  

In [0]:
%sql
-- Query returns the appropriate results with data for customer_id = 51 and long value for mall_pin_code in purchase_details.
-- This means manual Type widening worked for struct type as well.

SELECT
  *
FROM
  delta_catalog.delta_db.invoices_se
ORDER BY
  customer_id;

In [0]:
%sql
ALTER TABLE
  delta_catalog.delta_db.invoices_se
ADD COLUMN
  purchase_details.store_loc STRING;

In [0]:
%sql
INSERT INTO delta_catalog.delta_db.invoices_se
VALUES(52, 'I0002', 100, '2026-05-19', 33, 'Credit Card', "24", "Mall of Istanbul", struct(012345, 12345, "Istanbul"));  

In [0]:
%sql
-- Query returns the appropriate results with data for customer_id = 52 and string value for store_loc in purchase_details with historical store_loc as nulls.
-- This means adding new column(schema evolution) worked for struct type as well.

SELECT
  *
FROM
  delta_catalog.delta_db.invoices_se
ORDER BY
  customer_id;

#### Automatic Nested Structure Evolution

##### Using INSERT INTO (Need User-Managed Cluster)

In [0]:
%sql
DESCRIBE EXTENDED delta_catalog.delta_db.invoices_se;

In [0]:
%sql
-- Enable Column Mapping by Name so Delta can map structural keys natively
ALTER TABLE delta_catalog.delta_db.invoices_se
SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name');

In [0]:
%sql
-- Appending A Single Record To Verify Schema Evolution Nested Structure
-- Enable automatic schema evolution Setting on the table ('spark.databricks.delta.schema.autoMerge.enabled' = 'true'). 
INSERT INTO delta_catalog.delta_db.invoices_se
VALUES(88, 'I0003', 100, '2026-04-19', 33, 'Credit Card', "24", "Mall of Istanbul", 
named_struct( 'mall_pin_code',012345,
 'store_code', 12345,
 'store_loc', "Istanbul",
 'staff_id', "SID1235"
 )); 


##### Using MERGE INTO (Works On Serverless and User-Managed Cluster Both)

In [0]:
%sql
MERGE WITH SCHEMA EVOLUTION INTO delta_catalog.delta_db.invoices_se AS target
USING (
  SELECT 
    cast(54 AS bigint) AS customer_id,  -- Kept your bigint fix intact
    'I0003' AS invoice_no, 
    100.0 AS price, 
    '2026-04-19' AS invoice_date, 
    33 AS quantity, 
    'Credit Card' AS payment_method, 
    24 AS age, 
    'Mall of Istanbul' AS shopping_mall,
    named_struct(
      'mall_pin_code', 125,
      'store_code', 12345,
      'store_loc', 'Istanbul',
      'staff_id', 'SID1235'
    ) AS purchase_details  -- Giving the new struct an explicit column name
) AS source
ON target.customer_id = source.customer_id
WHEN NOT MATCHED THEN
  INSERT *;

In [0]:
%sql
-- by mistake added so deleted
-- DELETE FROM delta_catalog.delta_db.invoices_se WHERE customer_id = 53;
-- ALTER TABLE delta_catalog.delta_db.invoices_se
-- DROP COLUMN shopping_mall_details;

In [0]:
%sql
-- Query returns the appropriate results with data for customer_id = 54 and new added value for staff_id in purchase_details with historical staff_id as nulls.
-- This means aotumatic schema evolution worked for nested struct type as well.

SELECT
  *
FROM
  delta_catalog.delta_db.invoices_se
ORDER BY
  customer_id;

In [0]:
%sql
DESCRIBE HISTORY delta_catalog.delta_db.invoices_se;

### Scenario 4: Column Position Changes

In [0]:
%sql
-- ALTER TABLE delta_catalog.delta_db.invoices_se ADD COLUMN (gender STRING FIRST);
ALTER TABLE delta_catalog.delta_db.invoices_se ADD COLUMN (gender STRING AFTER price);

-- ALTER TABLE delta_catalog.delta_db.invoices_se
-- DROP COLUMN gender;

In [0]:
%sql
SELECT
  *
FROM
  delta_catalog.delta_db.invoices_se
LIMIT 1;

In [0]:
%sql
INSERT INTO delta_catalog.delta_db.invoices_se
  SELECT
    customer_id,
    invoice_no,
    price,
    gender,
    invoice_date,
    quantity,
    payment_method,
    age,
    shopping_mall,
    Null AS purchase_details
  FROM
    PARQUET.`abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/invoices/invoices_1_100.parquet`
  WHERE
    customer_id BETWEEN 60 AND 65;

In [0]:
%sql
-- Query returns the appropriate results with data for customer_id = 60-65 and new Column gender
-- at right position.
-- This means Column Position Change worked.

SELECT
  *
FROM
  delta_catalog.delta_db.invoices_se
ORDER BY
  customer_id;

In [0]:
%sql
DESCRIBE HISTORY delta_catalog.delta_db.invoices_se;